# Decomposition Loss Weight Sweep — Local Windows

Same experiments as `colab_train_sample_eval2.ipynb` but runs locally.  
Requires the project to be at `c:\Users\ameli\Desktop\TezBaselines\MyCode`.

In [ ]:
import os, sys
from pathlib import Path

REPO_PATH = r'c:\Users\ameli\Desktop\TezBaselines\MyCode'
assert Path(REPO_PATH).exists(), f'Not found: {REPO_PATH}'

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import wandb, os
os.environ['WANDB_API_KEY'] = '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'

_api_key = os.environ.get('WANDB_API_KEY')
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()

print('wandb version:', wandb.__version__)

In [ ]:
import torch, gc, importlib
from pathlib import Path

import train_with_mode as _twm
importlib.reload(_twm)
from train_with_mode import train

from evaluate_unified import evaluate

# ── Shared settings ───────────────────────────────────────────────────────────
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
WANDB_PROJECT = 'diffusion-timeseries'
NUM_WORKERS   = 0     # must be 0 on Windows
SEED          = 42
BATCH_SIZE    = 64

# Small model config
HIDDEN_DIM  = 64
NUM_LAYERS  = 3
NUM_EPOCHS  = 1000
LR          = 1e-4

# Inline metric settings — DiffusionTS protocol
EVAL_METRICS_EVERY  = 250
N_METRIC_ITERATIONS = 5

# ── Experiment list ───────────────────────────────────────────────────────────
EXPERIMENTS = [

    # ── FFT loss sweep ────────────────────────────────────────────────────────
    dict(name='fft_w010_local', group='sweep_fft',
         fft_weight=0.10, trend_weight=0.0, season_weight=0.0),
    dict(name='fft_w025_local', group='sweep_fft',
         fft_weight=0.25, trend_weight=0.0, season_weight=0.0),
    dict(name='fft_w050_local', group='sweep_fft',
         fft_weight=0.50, trend_weight=0.0, season_weight=0.0),
    dict(name='fft_w100_local', group='sweep_fft',
         fft_weight=1.00, trend_weight=0.0, season_weight=0.0),

    # ── Trend loss sweep ──────────────────────────────────────────────────────
    dict(name='trend_w010_local', group='sweep_trend',
         fft_weight=0.0, trend_weight=0.10, season_weight=0.0),
    dict(name='trend_w025_local', group='sweep_trend',
         fft_weight=0.0, trend_weight=0.25, season_weight=0.0),
    dict(name='trend_w050_local', group='sweep_trend',
         fft_weight=0.0, trend_weight=0.50, season_weight=0.0),
    dict(name='trend_w100_local', group='sweep_trend',
         fft_weight=0.0, trend_weight=1.00, season_weight=0.0),

    # ── Season loss sweep ─────────────────────────────────────────────────────
    dict(name='season_w010_local', group='sweep_season',
         fft_weight=0.0, trend_weight=0.0, season_weight=0.10),
    dict(name='season_w025_local', group='sweep_season',
         fft_weight=0.0, trend_weight=0.0, season_weight=0.25),
    dict(name='season_w050_local', group='sweep_season',
         fft_weight=0.0, trend_weight=0.0, season_weight=0.50),
    dict(name='season_w100_local', group='sweep_season',
         fft_weight=0.0, trend_weight=0.0, season_weight=1.00),
]

# ── Print plan ────────────────────────────────────────────────────────────────
print(f'Device  : {DEVICE}')
print(f'Total   : {len(EXPERIMENTS)} experiments')
print(f'Protocol: DiffusionTS — full dataset, disc=2000, pred=5000, {N_METRIC_ITERATIONS} runs\n')
print(f"  {'#':<4} {'Name':<18} {'Group':<14} {'fft':>5} {'trend':>6} {'season':>7}")
print('  ' + '-' * 56)
for i, exp in enumerate(EXPERIMENTS):
    print(f"  {i+1:<4} {exp['name']:<18} {exp['group']:<14}"
          f" {exp['fft_weight']:>5.2f} {exp['trend_weight']:>6.2f} {exp['season_weight']:>7.2f}")
print()

# ── Run loop ──────────────────────────────────────────────────────────────────
for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}]  {exp['name']}  (group: {exp['group']})")
    print(f"{'='*70}\n")

    ckpt_dir  = f"output/ckpt_{exp['name']}"
    best_ckpt = f"{ckpt_dir}/best_model.pt"

    # ── Train ─────────────────────────────────────────────────────────────────
    try:
        train(
            mode               = 'decomposition',
            device             = DEVICE,
            hidden_dim         = HIDDEN_DIM,
            num_layers         = NUM_LAYERS,
            num_epochs         = NUM_EPOCHS,
            lr                 = LR,
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp['name'],
            wandb_group        = exp['group'],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            img_pred_objective = 'pred_x0',
            img_loss_type      = 'l1',
            fft_weight         = exp['fft_weight'],
            trend_weight       = exp['trend_weight'],
            season_weight      = exp['season_weight'],
            checkpoint_dir     = ckpt_dir,
        )
    except Exception:
        import traceback
        print(f"\n!!! TRAINING FAILED: {exp['name']}")
        traceback.print_exc()
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Post-training evaluation ───────────────────────────────────────────────
    if Path(best_ckpt).exists():
        print(f"\n--- Post-training evaluation: {exp['name']} ---")
        try:
            evaluate(
                mode               = 'decomposition',
                checkpoint_path    = best_ckpt,
                device             = DEVICE,
                num_samples        = 256,
                n_metric_iterations= N_METRIC_ITERATIONS,
                compute_context_fid= True,
                use_wandb          = True,
                wandb_project      = WANDB_PROJECT,
                wandb_run_name     = exp['name'],
                wandb_group        = exp['group'],
                output_dir         = ckpt_dir,
                hidden_dim         = HIDDEN_DIM,
                num_layers         = NUM_LAYERS,
            )
        except Exception:
            import traceback
            print(f"\n!!! EVALUATION FAILED: {exp['name']}")
            traceback.print_exc()
    else:
        print(f"   [skip eval] best_model.pt not found at {best_ckpt}")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*70)
print('  ALL EXPERIMENTS COMPLETE')
print('='*70)